In [1]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow import keras
from skimage.metrics import structural_similarity as ssim
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Dense, Dropout
import cv2
from tqdm import tqdm
import tqdm as notebook_tqdm
import imutils
import pywt

2026-02-27 16:54:45.881023: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-27 16:54:46.437419: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-27 16:54:48.539716: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Discreet Wavelet Transform version of the vortex function
def wavelet_distillation(distilled_image):
    # Perform a 2D Discrete Wavelet Transform
    # 'haar' is great for detecting intensity jumps (tumor boundaries)
    coeffs2 = pywt.dwt2(distilled_image, 'haar')
    LL, (LH, HL, HH) = coeffs2
    
    # LL is the 'Approximation' (the general shape)
    # LH, HL, HH are 'Details' (the edges/texture)
    
    # We can stack these to create a multi-resolution feature map
    # This gives the model 4x the 'logic' to work with
    wavelet_stack = np.block([
        [LL, LH],
        [HL, HH]
    ])
    
    return wavelet_stack



# Fast Fourier Transform version of the vortex function
def myfft2(A): 
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(A)))

In [3]:
def generate_vortex_map(f, A, m, dim=64, xmax=10, flambda=80):
    """
    Applies a vortex phase of charge 'm' to a batch of images 'A'.
    
    Parameters:
    A (ndarray): Input image batch of shape [num_images, dim, dim]
    m (int): Topological charge (1, 2, 3, etc.)
    dim (int): Pixel resolution of the grid
    xmax (float): Physical limit of the f-plane
    flambda (float): Lens parameter
    
    Returns:
    ndarray: Processed and normalized images of shape [num_images, dim, dim]
    """
    # 1. Setup Coordinates
    dx = 2 * xmax / dim
    x = np.linspace(-xmax, xmax - dx, dim)
    X, Y = np.meshgrid(x, x)
    R2 = (X**2 + Y**2)
    T = np.arctan2(Y, X)

    # 2. Create the LG Mask (Lens + Vortex Phase)
    Lens = np.exp(-1j * R2 / (flambda / 10))
    Vortex = np.exp(1j * m * T)
    LG_mask = Lens * Vortex

    # 3. Process the Batch
    num_images = A.shape[0]
    out = np.zeros_like(A, dtype=np.float64)
    
    for i in range(num_images):
        # Apply phase modulation and distillation 
        # (Assumes wavelet_distillation is defined globally)
        phase_mod = np.exp(1j * A[i] * np.pi * 2)
        distilled = f(phase_mod * LG_mask)
        
        # Calculate Magnitude
        A_mag = np.abs(distilled)
        
        # Normalize to [-1, 1] for NN compatibility
        a_min = np.amin(A_mag)
        a_diff = np.amax(A_mag) - a_min
        
        if a_diff != 0:
            out[i] = (A_mag - a_min) / a_diff * 2 - 1
        else:
            out[i] = A_mag - a_min

    return out

In [5]:
def base_model():
    model = Sequential()
    model.add(Dense(dim ** 2, input_dim= dim ** 2, activation='linear', use_bias=False))  
    model.compile(loss='MSE', optimizer='adam', metrics=['accuracy'])
    return model

In [7]:
# Train and save the model
np.random.seed(101)
imax = 10000
dim = 64   # Test images are CiFAR10. CiFAR10 is 32x32

model = base_model()

for d in [2.0, 4.0, 7.0, 10.0, 12.0]:
    # Make the datasets
    # 1. Setup coordinates
    x=np.linspace(-10,10,dim)
    X,Y=np.meshgrid(x,x)
    R2 = X**2 + Y**2


    dataset=np.zeros((imax,dim,dim))
    dist = d


    for j in range(imax):
        # Generate the random phase (the internal "texture")
        RandP = np.random.uniform(size = [dim,dim]) # Noise

        dataset[j] = np.abs(np.fft.fft2(R2/dist*np.exp(-R2/dist)*np.exp(1j*2*np.pi*RandP)))


    # 1. Normalize the ground truth dataset [0, 1]
    # imax is your total number of images
    gt = np.zeros((imax, dim, dim))

    for j in range(imax):
        # Local normalization for each speckle pattern
        d_min = np.amin(dataset[j])
        d_max = np.amax(dataset[j] - d_min)
        
        if d_max != 0:
            gt[j] = (dataset[j] - d_min) / d_max
        else:
            gt[j] = dataset[j] - d_min

    
    for f_i in [20, 40, 60, 80, 120, 160, 200, 250]:
        # 1. Process through the vortex function for m=1
        # (Assuming 'generate_vortex_map' is the function from the previous step)
        X = generate_vortex_map(wavelet_distillation, gt, m=1, dim=64, flambda=f_i)

        # 2. Reshape for the Neural Network (Flattening the spatial dimensions)
        # This converts [imax, 64, 64] -> [imax, 4096]
        X_train = X.reshape(X.shape[0], -1)
        Y_train = gt.reshape(gt.shape[0], -1)

        model.fit(X_train, Y_train, validation_split=0.01, shuffle = True, verbose = 1, batch_size = 784*60, epochs=5000)

        file_path = '/mnt/c/Users/Jet/Documents/PhD Data/Machine Vision/Reconstruction Models/Wavelet/FineTune/'
        model.save(file_path + f'dist{d}_flambad{f_i}.keras')

/home/jet/anaconda3/envs/MV/lib/python3.13/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0101e-04 - loss: 0.4903 - val_accuracy: 0.0000e+00 - val_loss: 1.7161
Epoch 2/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 904ms/step - accuracy: 4.0404e-04 - loss: 1.7023 - val_accuracy: 0.0000e+00 - val_loss: 0.4531
Epoch 3/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 2.0202e-04 - loss: 0.4497 - val_accuracy: 0.0000e+00 - val_loss: 0.5582
Epoch 4/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 974ms/step - accuracy: 3.0303e-04 - loss: 0.5548 - val_accuracy: 0.0000e+00 - val_loss: 0.9689
Epoch 5/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 925ms/step - accuracy: 2.0202e-04 - loss: 0.9631 - val_accuracy: 0.0000e+00 - val_loss: 0.6548
Epoch 6/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 957ms/step - accuracy: 1.0101e-04 - loss: 0.6526 - val_accuracy: 0.0000e+00 - val_loss: 0.2426
Epoch 7/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 931ms/step - accuracy: 2.0202e-04 - loss: 0.2439 - val_accuracy: 0.0000e+00 - val_loss: 0.2790
Epoch 8/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 913ms/step - 

KeyboardInterrupt: 